# Perdidas que generan transicion y tiro

Notebook sencillo para construir un dataset de regresion logistica.

La unidad de analisis ya no es una recuperacion aislada. Ahora es:

> una posesion de un equipo que termina y da paso a una posesion rival en juego abierto.

Para cada perdida guardamos:

- equipo que pierde el balon;
- equipo que recupera/transiciona;
- posicion estimada de perdida;
- accion que explica la perdida (`Block`, `Interception`, `Duel`, `Miscontrol`, etc.);
- si el rival tira en los siguientes 5 segundos;
- minuto, marcador e igualdad numerica.

In [114]:
import warnings

import numpy as np
import pandas as pd
from statsbombpy import sb

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 100)


## 1. Elegir partido

Usamos un partido de Bundesliga 2023/2024. Si quieres otro partido, cambia `MATCH_ID`.

In [126]:
# Bundesliga 2023/2024
COMPETITION_ID = 9
SEASON_ID = 281

# None = usa el primer partido disponible de esa liga/temporada.
# Tambien puedes poner un match_id concreto.
MATCH_ID = None

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID).copy()
matches = matches.sort_values(["match_date", "match_id"]).reset_index(drop=True)

if MATCH_ID is None:
    MATCH_ID = int(matches.loc[22, "match_id"])

match_info = matches[matches["match_id"] == MATCH_ID].copy()
home_team = match_info.iloc[0]["home_team"]
away_team = match_info.iloc[0]["away_team"]

print(f"MATCH_ID usado: {MATCH_ID}")
display(match_info[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score"]])


MATCH_ID usado: 3895250


,match_id,match_date,home_team,away_team,home_score,away_score
22,3895250,2024-02-23,Bayer Leverkusen,FSV Mainz 05,2,1


## 2. Cargar eventos

Si falta alguna columna, la creamos con `pd.NA`. Esto evita que el notebook se rompa entre partidos con columnas distintas.

In [131]:
COLS_NEEDED = [
    "id", "index", "period", "timestamp", "minute", "second",
    "type", "play_pattern", "possession", "possession_team", "team", "player",
    "location", "out", "under_pressure", "counterpress",
    "shot_outcome", "shot_statsbomb_xg",
    "ball_recovery_recovery_failure",
    "ball_recovery_offensive",
    "block_deflection", "block_offensive", "block_save_block",
    "interception_outcome",
    "duel_type", "duel_outcome",
    "50_50_outcome",
    "clearance_aerial_won", "clearance_body_part",
    "pass_outcome", "pass_end_location", "dribble_outcome", "ball_receipt_outcome",
    "foul_committed_card", "bad_behaviour_card",
]


def has_xy(value):
    return isinstance(value, (list, tuple)) and len(value) >= 2


def get_x(value):
    return float(value[0]) if has_xy(value) else np.nan


def get_y(value):
    return float(value[1]) if has_xy(value) else np.nan


def get_end_x(value):
    return float(value[0]) if has_xy(value) else np.nan


def get_end_y(value):
    return float(value[1]) if has_xy(value) else np.nan


def prepare_events(raw_events):
    events = raw_events.copy()

    # Rellenamos columnas que no existan en este partido.
    for col in COLS_NEEDED:
        if col not in events.columns:
            events[col] = pd.NA

    events = events[COLS_NEEDED].copy()

    # Orden estable de eventos dentro del partido.
    fallback_order = pd.Series(np.arange(len(events)), index=events.index)
    events["event_order"] = pd.to_numeric(events["index"], errors="coerce").fillna(fallback_order)

    # Coordenadas StatsBomb: location = [x, y].
    events["x"] = events["location"].apply(get_x)
    events["y"] = events["location"].apply(get_y)
    events["pass_end_x"] = events["pass_end_location"].apply(get_end_x)
    events["pass_end_y"] = events["pass_end_location"].apply(get_end_y)

    events["period"] = pd.to_numeric(events["period"], errors="coerce").fillna(0).astype(int)
    events["minute"] = pd.to_numeric(events["minute"], errors="coerce")
    events["second"] = pd.to_numeric(events["second"], errors="coerce")
    events["event_seconds"] = events["minute"].fillna(0) * 60 + events["second"].fillna(0)
    events["possession"] = pd.to_numeric(events["possession"], errors="coerce")
    events["match_id"] = int(MATCH_ID)

    return events.sort_values(["period", "event_order"]).reset_index(drop=True)


raw_events = sb.events(match_id=MATCH_ID)
events = prepare_events(raw_events)

print(f"Eventos cargados: {len(events):,}")
events.head()


Eventos cargados: 3,826


,id,index,period,timestamp,minute,second,type,play_pattern,possession,possession_team,team,player,location,out,under_pressure,counterpress,shot_outcome,shot_statsbomb_xg,ball_recovery_recovery_failure,ball_recovery_offensive,block_deflection,block_offensive,block_save_block,interception_outcome,duel_type,duel_outcome,50_50_outcome,clearance_aerial_won,clearance_body_part,pass_outcome,pass_end_location,dribble_outcome,ball_receipt_outcome,foul_committed_card,bad_behaviour_card,event_order,x,y,pass_end_x,pass_end_y,event_seconds,match_id
0,63d94113-a4e0-45a3-b4de-87b8d27d5524,1,1,00:00:00.000,0,0,Starting XI,Regular Play,1,Bayer Leverkusen,Bayer Leverkusen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,0,3895250
1,9f12347c-3391-4a36-9d43-4f72a39c2447,2,1,00:00:00.000,0,0,Starting XI,Regular Play,1,Bayer Leverkusen,FSV Mainz 05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,0,3895250
2,398b7370-8ecd-40a0-9f93-85a9cf92adb4,3,1,00:00:00.000,0,0,Half Start,Regular Play,1,Bayer Leverkusen,FSV Mainz 05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,0,3895250
3,213df70e-f903-4676-b38b-4d61ba9902f7,4,1,00:00:00.000,0,0,Half Start,Regular Play,1,Bayer Leverkusen,Bayer Leverkusen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,0,3895250
4,2c8959ba-eede-48db-a7c3-c06853b94042,5,1,00:00:01.186,0,1,Pass,From Kick Off,2,Bayer Leverkusen,Bayer Leverkusen,Jonas Hofmann,"[61.0, 40.1]",NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,"[39.7, 48.5]",NaN,NaN,NaN,NaN,5,61.0,40.1,39.7,48.5,1,3895250


## 3. Concepto: perdida vs recuperacion

Una **perdida** se mira desde el equipo que tenia la posesion.

Una **recuperacion** se mira desde el rival que roba o recoge el balon.

Para la regresion queremos una fila por perdida:

```text
equipo A ataca -> equipo A pierde -> equipo B transiciona -> equipo B tira o no tira en 5s
```

Por eso partimos de cambios de posesion entre equipos, no de eventos sueltos.

In [132]:
OPEN_PLAY_NEXT_PATTERNS = {"Regular Play", "From Counter"}
SHOT_WINDOW_SECONDS = 5


def is_true(value):
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.lower() in {"true", "1", "yes", "si", "sí"}
    return bool(value)


def possession_meta(events):
    """Una fila por posesion: equipo, inicio y final."""
    return (
        events.dropna(subset=["possession"])
        .sort_values(["period", "event_order"])
        .groupby(["match_id", "period", "possession"], observed=True)
        .agg(
            possession_team=("possession_team", "first"),
            play_pattern=("play_pattern", "first"),
            start_order=("event_order", "min"),
            end_order=("event_order", "max"),
            start_seconds=("event_seconds", "min"),
            end_seconds=("event_seconds", "max"),
        )
        .reset_index()
        .sort_values(["period", "start_order"])
        .reset_index(drop=True)
    )


meta = possession_meta(events)
meta.head(10)


,match_id,period,possession,possession_team,play_pattern,start_order,end_order,start_seconds,end_seconds
0,3895250,1,1,Bayer Leverkusen,Regular Play,1,4,0,0
1,3895250,1,2,Bayer Leverkusen,From Kick Off,5,26,1,22
2,3895250,1,3,Bayer Leverkusen,From Throw In,27,33,25,28
3,3895250,1,4,FSV Mainz 05,From Throw In,34,60,50,66
4,3895250,1,5,Bayer Leverkusen,From Throw In,61,91,77,106
5,3895250,1,6,Bayer Leverkusen,From Free Kick,92,102,150,160
6,3895250,1,7,FSV Mainz 05,From Kick Off,103,106,214,221
7,3895250,1,8,Bayer Leverkusen,From Keeper,107,115,221,245
8,3895250,1,9,FSV Mainz 05,Regular Play,116,128,245,257
9,3895250,1,10,Bayer Leverkusen,From Free Kick,129,172,288,313


## 4. Elegir la accion que explica la perdida

Para cada cambio de posesion en juego abierto:

1. `losing_team` es el `possession_team` de la posesion que acaba.
2. `transition_team` es el `possession_team` de la siguiente posesion.
3. Buscamos cerca del cambio una accion que explique la perdida.

Priorizamos acciones defensivas del equipo que roba: `Block`, `Interception`, `Duel/Tackle`, `50/50`, `Clearance`.

Si no hay una accion defensiva dentro de la posesion que acaba, aceptamos un `Ball Recovery` solo cuando es el primer evento de la nueva posesion y la posesion anterior contiene una señal clara de perdida (`Ball Receipt* Incomplete`, `Miscontrol`, `Dispossessed`, `Pass Incomplete`, etc.). Asi capturamos balones sueltos sin usar ventanas arbitrarias.

Si no encontramos accion defensiva clara, usamos la ultima accion de ataque del equipo que pierde: `Miscontrol`, `Dispossessed`, `Dribble`, `Pass`, etc.

In [133]:
DEFENSIVE_PRIORITY = {
    "Block": 1,
    "Interception": 2,
    "Duel": 3,
    "50/50": 4,
    "Clearance": 5,
}

ATTACKING_PRIORITY = {
    "Miscontrol": 1,
    "Dispossessed": 2,
    "Dribble": 3,
    "Pass": 4,
    "Carry": 5,
    "Ball Receipt*": 6,
}

VALID_TACKLE_OUTCOMES = {"Won", "Success", "Success In Play"}
VALID_INTERCEPTION_OUTCOMES = {"Won", "Success", "In Play", "Success In Play"}
VALID_50_50_OUTCOMES = {"Won", "Success To Team"}
INCOMPLETE_PASS_OUTCOMES = {"Incomplete", "Out", "Pass Offside", "Unknown", "Injury Clearance"}
INCOMPLETE_DRIBBLE_OUTCOMES = {"Incomplete"}
INCOMPLETE_RECEIPT_OUTCOMES = {"Incomplete"}
SHOT_END_TYPES = {"Shot", "Goal Keeper"}


def mirror_x(x):
    return PITCH_LENGTH - x if pd.notna(x) else np.nan


def mirror_y(y):
    return PITCH_WIDTH - y if pd.notna(y) else np.nan


PITCH_LENGTH = 120
PITCH_WIDTH = 80


def is_restart_or_out(previous_events, next_possession):
    """Excluye cambios por banda, faltas, corners, saques, etc."""
    if next_possession["play_pattern"] not in OPEN_PLAY_NEXT_PATTERNS:
        return True

    if previous_events.empty:
        return True

    last_event = previous_events.sort_values("event_order").iloc[-1]

    # Si la posesion anterior contiene un tiro, no es una perdida de las que
    # queremos modelar. Es una posesion que ya termino atacando.
    if (previous_events["type"] == "Shot").any():
        return True

    # Si acaba con accion de portero, tambien suele ser una continuacion natural
    # tras tiro, parada, recepcion del portero o reinicio.
    if str(last_event.get("type", "")) == "Goal Keeper":
        return True

    if is_true(last_event.get("out", pd.NA)):
        return True

    if str(last_event.get("pass_outcome", "")) in {"Out", "Pass Offside", "Injury Clearance"}:
        return True

    return False


def has_attacking_loss_signal(event):
    """True si el evento del equipo que ataca describe perdida o mal control."""
    event_type = event.get("type")

    if event_type in {"Miscontrol", "Dispossessed"}:
        return True
    if event_type == "Ball Receipt*":
        return str(event.get("ball_receipt_outcome")) in INCOMPLETE_RECEIPT_OUTCOMES
    if event_type == "Dribble":
        return str(event.get("dribble_outcome")) in INCOMPLETE_DRIBBLE_OUTCOMES
    if event_type == "Pass":
        return str(event.get("pass_outcome")) in INCOMPLETE_PASS_OUTCOMES
    if event_type == "Duel":
        return "Lost" in str(event.get("duel_outcome"))
    return False


def attacking_loss_candidates(previous_events, losing_team):
    """Eventos del equipo que pierde que explican la perdida."""
    candidates = previous_events[
        (previous_events["team"] == losing_team)
        & (previous_events["type"].isin(ATTACKING_PRIORITY) | (previous_events["type"] == "Duel"))
        & previous_events["x"].notna()
        & previous_events["y"].notna()
    ].copy()

    if candidates.empty:
        return candidates

    candidates = candidates[candidates.apply(has_attacking_loss_signal, axis=1)].copy()
    if candidates.empty:
        return candidates

    candidates["priority"] = candidates["type"].map(ATTACKING_PRIORITY).fillna(7)
    return candidates


def first_next_possession_recovery(events, next_possession, transition_team):
    """Ball Recovery valido solo si abre la nueva posesion."""
    next_events = events[
        (events["period"] == int(next_possession["period"]))
        & (events["possession"] == next_possession["possession"])
    ].sort_values("event_order").copy()

    if next_events.empty:
        return pd.Series(dtype="object")

    first_event = next_events.iloc[0]
    if (
        first_event.get("type") == "Ball Recovery"
        and first_event.get("team") == transition_team
        and pd.isna(first_event.get("ball_recovery_recovery_failure"))
        and pd.notna(first_event.get("x"))
        and pd.notna(first_event.get("y"))
    ):
        return first_event

    return pd.Series(dtype="object")


def event_location_for_loss(event):
    """Punto fisico de perdida para el evento elegido.

    Para un pase incompleto, preferimos el destino del pase si existe.
    Para el resto usamos la localizacion del evento.
    """
    if event.get("type") == "Pass" and pd.notna(event.get("pass_end_x")) and pd.notna(event.get("pass_end_y")):
        return float(event["pass_end_x"]), float(event["pass_end_y"])
    return float(event["x"]), float(event["y"])


def filter_valid_defensive_candidates(candidates):
    """Evita meter duelos/intercepciones que no ganan realmente la pelota."""
    if candidates.empty:
        return candidates

    valid = []
    for _, event in candidates.iterrows():
        event_type = event["type"]

        if event_type == "Duel":
            valid.append(
                event.get("duel_type") == "Tackle"
                and str(event.get("duel_outcome")) in VALID_TACKLE_OUTCOMES
            )
        elif event_type == "Interception":
            valid.append(
                str(event.get("interception_outcome")) in VALID_INTERCEPTION_OUTCOMES
                or pd.isna(event.get("interception_outcome"))
            )
        elif event_type == "50/50":
            valid.append(str(event.get("50_50_outcome")) in VALID_50_50_OUTCOMES)
        else:
            valid.append(True)

    return candidates[pd.Series(valid, index=candidates.index)].copy()


def choose_turnover_event(events, current_possession, next_possession):
    """Elige el evento que mejor representa la perdida."""
    losing_team = current_possession["possession_team"]
    transition_team = next_possession["possession_team"]
    period = int(current_possession["period"])

    # Eventos de la posesion que acaba.
    previous_events = events[
        (events["period"] == period)
        & (events["possession"] == current_possession["possession"])
    ].copy()

    # Primero buscamos acciones defensivas del equipo que transiciona, pero
    # SOLO dentro de la posesion que pierde el rival.
    defensive_candidates = previous_events.copy()
    defensive_candidates = defensive_candidates[
        (defensive_candidates["team"] == transition_team)
        & (defensive_candidates["type"].isin(DEFENSIVE_PRIORITY))
        & defensive_candidates["x"].notna()
        & defensive_candidates["y"].notna()
    ].copy()
    defensive_candidates = filter_valid_defensive_candidates(defensive_candidates)

    if not defensive_candidates.empty:
        defensive_candidates["priority"] = defensive_candidates["type"].map(DEFENSIVE_PRIORITY)
        event = defensive_candidates.sort_values(["priority", "event_order"]).iloc[0]
        source = "defensive_event"
    else:
        attacking_candidates = attacking_loss_candidates(previous_events, losing_team)

        # Ball Recovery entra solo si la posesion previa tiene una perdida clara
        # y el Ball Recovery abre la nueva posesion.
        next_recovery = first_next_possession_recovery(events, next_possession, transition_team)
        if not attacking_candidates.empty and not next_recovery.empty:
            event = next_recovery
            source = "next_possession_recovery"
        elif not attacking_candidates.empty:
            # Si no hay recuperacion inicial, usamos la ultima señal de perdida
            # del equipo que atacaba.
            event = attacking_candidates.sort_values(["event_order", "priority"], ascending=[False, True]).iloc[0]
            source = "attacking_event"
        else:
            return None, previous_events

    event_x, event_y = event_location_for_loss(event)

    return {
        "turnover_event_source": source,
        "turnover_event_id": event.get("id", pd.NA),
        "turnover_action_type": event.get("type", pd.NA),
        "turnover_player": event.get("player", pd.NA),
        "turnover_event_team": event.get("team", pd.NA),
        "turnover_event_order": float(event["event_order"]),
        "turnover_event_seconds": float(event["event_seconds"]),
        "turnover_x": event_x,
        "turnover_y": event_y,
        "raw_event_x": float(event["x"]),
        "raw_event_y": float(event["y"]),
        "pass_outcome": event.get("pass_outcome", pd.NA),
        "dribble_outcome": event.get("dribble_outcome", pd.NA),
        "ball_receipt_outcome": event.get("ball_receipt_outcome", pd.NA),
        "duel_type": event.get("duel_type", pd.NA),
        "duel_outcome": event.get("duel_outcome", pd.NA),
        "interception_outcome": event.get("interception_outcome", pd.NA),
        "50_50_outcome": event.get("50_50_outcome", pd.NA),
        "clearance_aerial_won": event.get("clearance_aerial_won", pd.NA),
        "clearance_body_part": event.get("clearance_body_part", pd.NA),
        "ball_recovery_offensive": event.get("ball_recovery_offensive", pd.NA),
        "block_deflection": event.get("block_deflection", pd.NA),
        "block_offensive": event.get("block_offensive", pd.NA),
        "block_save_block": event.get("block_save_block", pd.NA),
        "under_pressure": event.get("under_pressure", pd.NA),
        "counterpress": event.get("counterpress", pd.NA),
    }, previous_events


## 5. Construir dataset de perdidas

Guardamos dos pares de coordenadas:

- `loss_x`, `loss_y`: punto visto desde el equipo que pierde el balon. Son las variables principales para modelar riesgo de perdida.
- `transition_x`, `transition_y`: el mismo punto visto desde el equipo que roba/transiciona. Sirven para debug y mapas desde el equipo que ataca despues.

Si el evento elegido lo hace el rival (`defensive_event` o `next_possession_recovery`), `turnover_x/y` ya estan en perspectiva del equipo que transiciona y los espejamos para obtener `loss_x/y`.

Si el evento elegido lo hace el equipo que pierde (`attacking_event`), `turnover_x/y` ya son `loss_x/y` y los espejamos para obtener `transition_x/y`.

In [134]:
def first_shot_after_turnover(events, transition_team, next_possession, t0_seconds, window_seconds=5):
    """Primer tiro del equipo que transiciona en su siguiente posesion y dentro de la ventana."""
    shots = events[
        (events["period"] == int(next_possession["period"]))
        & (events["possession"] == next_possession["possession"])
        & (events["team"] == transition_team)
        & (events["type"] == "Shot")
    ].copy()

    if shots.empty:
        return {
            "shot_within_5s": 0,
            "seconds_to_shot": np.nan,
            "shot_player": pd.NA,
            "shot_outcome": pd.NA,
            "shot_statsbomb_xg": np.nan,
        }

    shots["seconds_to_shot"] = shots["event_seconds"] - t0_seconds
    shots = shots[
        (shots["seconds_to_shot"] >= 0)
        & (shots["seconds_to_shot"] <= window_seconds)
    ].sort_values(["seconds_to_shot", "event_order"])

    if shots.empty:
        return {
            "shot_within_5s": 0,
            "seconds_to_shot": np.nan,
            "shot_player": pd.NA,
            "shot_outcome": pd.NA,
            "shot_statsbomb_xg": np.nan,
        }

    shot = shots.iloc[0]
    return {
        "shot_within_5s": 1,
        "seconds_to_shot": float(shot["seconds_to_shot"]),
        "shot_player": shot.get("player", pd.NA),
        "shot_outcome": shot.get("shot_outcome", pd.NA),
        "shot_statsbomb_xg": shot.get("shot_statsbomb_xg", np.nan),
    }


rows = []

for i in range(len(meta) - 1):
    current_poss = meta.iloc[i]
    next_poss = meta.iloc[i + 1]

    # Solo nos interesan cambios de posesion dentro del mismo periodo.
    if current_poss["period"] != next_poss["period"]:
        continue

    losing_team = current_poss["possession_team"]
    transition_team = next_poss["possession_team"]

    # Si sigue el mismo equipo, no hay perdida entre equipos.
    if losing_team == transition_team:
        continue

    turnover_event, previous_events = choose_turnover_event(events, current_poss, next_poss)
    if turnover_event is None:
        continue

    # Excluimos cambios por banda/falta/reinicio. Queremos transiciones en juego.
    if is_restart_or_out(previous_events, next_poss):
        continue

    if turnover_event["turnover_event_source"] in {"defensive_event", "next_possession_recovery"}:
        loss_x = mirror_x(turnover_event["turnover_x"])
        loss_y = mirror_y(turnover_event["turnover_y"])
        transition_x = turnover_event["turnover_x"]
        transition_y = turnover_event["turnover_y"]
    else:
        loss_x = turnover_event["turnover_x"]
        loss_y = turnover_event["turnover_y"]
        transition_x = mirror_x(turnover_event["turnover_x"])
        transition_y = mirror_y(turnover_event["turnover_y"])

    shot_info = first_shot_after_turnover(
        events=events,
        transition_team=transition_team,
        next_possession=next_poss,
        # La transicion empieza cuando arranca la nueva posesion del rival,
        # no necesariamente en el evento defensivo que causo la perdida.
        t0_seconds=float(next_poss["start_seconds"]),
        window_seconds=SHOT_WINDOW_SECONDS,
    )

    rows.append({
        "match_id": int(current_poss["match_id"]),
        "period": int(current_poss["period"]),
        "losing_team": losing_team,
        "transition_team": transition_team,
        "lost_possession": current_poss["possession"],
        "transition_possession": next_poss["possession"],
        "lost_possession_play_pattern": current_poss["play_pattern"],
        "transition_play_pattern": next_poss["play_pattern"],
        "action_minute": round(float(next_poss["start_seconds"]) / 60, 2),
        "event_seconds": float(next_poss["start_seconds"]),
        "loss_x": loss_x,
        "loss_y": loss_y,
        "transition_x": transition_x,
        "transition_y": transition_y,
        **turnover_event,
        **shot_info,
    })

loss_transition_events = pd.DataFrame(rows)

print(f"Perdidas en juego detectadas: {len(loss_transition_events):,}")
print(f"Perdidas con tiro rival en <= {SHOT_WINDOW_SECONDS}s: {int(loss_transition_events['shot_within_5s'].sum()) if not loss_transition_events.empty else 0:,}")
loss_transition_events.head(20)


Perdidas en juego detectadas: 59
Perdidas con tiro rival en <= 5s: 2


,match_id,period,losing_team,transition_team,lost_possession,transition_possession,lost_possession_play_pattern,transition_play_pattern,action_minute,event_seconds,loss_x,loss_y,transition_x,transition_y,turnover_event_source,turnover_event_id,turnover_action_type,turnover_player,turnover_event_team,turnover_event_order,turnover_event_seconds,turnover_x,turnover_y,raw_event_x,raw_event_y,pass_outcome,dribble_outcome,ball_receipt_outcome,duel_type,duel_outcome,interception_outcome,50_50_outcome,clearance_aerial_won,clearance_body_part,ball_recovery_offensive,block_deflection,block_offensive,block_save_block,under_pressure,counterpress,shot_within_5s,seconds_to_shot,shot_player,shot_outcome,shot_statsbomb_xg
0,3895250,1,Bayer Leverkusen,FSV Mainz 05,8,9,From Keeper,Regular Play,4.08,245.0,7.0,66.0,113.0,14.0,attacking_event,bbd507c2-3e70-48ec-bbbb-006428267e1c,Ball Receipt*,Jonathan Tah,Bayer Leverkusen,115.0,245.0,7.0,66.0,7.0,66.0,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
1,3895250,1,Bayer Leverkusen,FSV Mainz 05,10,11,From Free Kick,Regular Play,5.22,313.0,38.8,46.6,81.2,33.4,defensive_event,f9db3df4-9bae-43f3-a735-bcb10c61f0f8,Block,Leandro Barreiro Martins,FSV Mainz 05,159.0,303.0,81.2,33.4,81.2,33.4,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
2,3895250,1,FSV Mainz 05,Bayer Leverkusen,13,14,From Free Kick,From Counter,6.30,378.0,113.4,38.6,6.6,41.4,defensive_event,4805e2a9-47ac-44b0-a338-0315fd96644d,Clearance,Alejandro Grimaldo García,Bayer Leverkusen,216.0,375.0,6.6,41.4,6.6,41.4,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,Right Foot,<NA>,<NA>,NaN,NaN,True,NaN,0,NaN,NaN,NaN,NaN
3,3895250,1,Bayer Leverkusen,FSV Mainz 05,14,15,From Counter,Regular Play,6.38,383.0,44.1,41.2,75.9,38.8,attacking_event,e05c5dd6-c915-48ae-8fd1-17ad2845fda5,Ball Receipt*,Florian Wirtz,Bayer Leverkusen,222.0,383.0,44.1,41.2,44.1,41.2,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,3895250,1,Bayer Leverkusen,FSV Mainz 05,19,20,From Throw In,Regular Play,9.20,552.0,73.2,60.9,46.8,19.1,attacking_event,a65e2051-fbe7-4a8c-8c0b-4e48ccfdf954,Ball Receipt*,Florian Wirtz,Bayer Leverkusen,286.0,552.0,73.2,60.9,73.2,60.9,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
5,3895250,1,Bayer Leverkusen,FSV Mainz 05,24,25,From Free Kick,Regular Play,12.13,728.0,92.1,77.6,27.9,2.4,attacking_event,78900831-6465-47d8-abf3-46ee93a86698,Ball Receipt*,Jeremie Frimpong,Bayer Leverkusen,421.0,728.0,92.1,77.6,92.1,77.6,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
6,3895250,1,FSV Mainz 05,Bayer Leverkusen,25,26,Regular Play,Regular Play,12.22,733.0,65.4,20.2,54.6,59.8,attacking_event,110e3a3b-f804-46da-a953-d25823754b9f,Ball Receipt*,Karim Onisiwo,FSV Mainz 05,432.0,733.0,65.4,20.2,65.4,20.2,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
7,3895250,1,Bayer Leverkusen,FSV Mainz 05,26,27,Regular Play,Regular Play,13.82,829.0,99.1,31.5,20.9,48.5,defensive_event,999c717e-88a6-499b-b766-9f597961ea26,Interception,Sepp van den Berg,FSV Mainz 05,537.0,827.0,20.9,48.5,20.9,48.5,NaN,NaN,NaN,NaN,NaN,Success In Play,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
8,3895250,1,Bayer Leverkusen,FSV Mainz 05,33,34,From Throw In,Regular Play,17.78,1067.0,104.1,40.5,15.9,39.5,next_possession_recovery,87d3efb4-c62f-4556-ade2-ec03da17e973,Ball Recovery,Anthony Caci,FSV Mainz 05,731.0,1067.0,15.9,39.5,15.9,39.5,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
9,3895250,1,FSV Mainz 05,Bayer Leverkusen,34,35,Regular Play,Regular Play,17.97,1078.0,76.4,11.9,43.6,68.1,attacking_event,a9a3a4f0-c8dd-495e-b320-e97ee0fbee88,Ball Receipt*,Jae-Sung Lee,FSV Mainz 05,742.0,1078.0,76.4,11.9,76.4,11.9,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


## 6. Contexto de marcador e igualdad numerica

El contexto se calcula desde el punto de vista del equipo que pierde el balon.

In [135]:
def opponent_of(team):
    if team == home_team:
        return away_team
    if team == away_team:
        return home_team
    return pd.NA


goals = (
    events[(events["type"] == "Shot") & (events["shot_outcome"] == "Goal")]
    [["period", "event_order", "event_seconds", "team"]]
    .sort_values(["period", "event_order"])
    .reset_index(drop=True)
)

red_card_values = {"Red Card", "Second Yellow", "Second Yellow Card"}
red_cards = events[
    events[["foul_committed_card", "bad_behaviour_card"]]
    .apply(lambda row: any(str(value) in red_card_values for value in row.dropna()), axis=1)
][["period", "event_order", "event_seconds", "team"]].copy()


def score_state_before(period, event_order, team):
    """Estado de marcador de un equipo antes de una accion."""
    rival = opponent_of(team)
    previous_goals = goals[
        (goals["period"] < period)
        | ((goals["period"] == period) & (goals["event_order"] < event_order))
    ]

    team_goals = int((previous_goals["team"] == team).sum())
    rival_goals = int((previous_goals["team"] == rival).sum())
    diff = team_goals - rival_goals

    if diff > 0:
        return "winning", diff
    if diff < 0:
        return "losing", diff
    return "drawing", diff


def numerical_equality_before(period, event_order, team):
    """Igualdad numerica antes de una accion, restando rojas previas."""
    rival = opponent_of(team)
    previous_reds = red_cards[
        (red_cards["period"] < period)
        | ((red_cards["period"] == period) & (red_cards["event_order"] < event_order))
    ]

    team_players = 11 - int((previous_reds["team"] == team).sum())
    rival_players = 11 - int((previous_reds["team"] == rival).sum())
    return team_players == rival_players, team_players, rival_players


context_rows = []
for _, loss in loss_transition_events.iterrows():
    score_state, score_diff = score_state_before(
        period=int(loss["period"]),
        event_order=float(loss["turnover_event_order"]),
        team=loss["losing_team"],
    )
    is_equal, losing_players, transition_players = numerical_equality_before(
        period=int(loss["period"]),
        event_order=float(loss["turnover_event_order"]),
        team=loss["losing_team"],
    )

    context_rows.append({
        "losing_team_score_state": score_state,
        "losing_team_score_diff": score_diff,
        "losing_team_players": losing_players,
        "transition_team_players": transition_players,
        # Positivo: el equipo que roba/transiciona tiene mas jugadores.
        # Negativo: el equipo que roba/transiciona tiene menos jugadores.
        "transition_team_player_diff": transition_players - losing_players,
        "is_numerical_equality": is_equal,
    })

context = pd.DataFrame(context_rows)
model_dataset = pd.concat([loss_transition_events.reset_index(drop=True), context], axis=1)

model_dataset.head()


,match_id,period,losing_team,transition_team,lost_possession,transition_possession,lost_possession_play_pattern,transition_play_pattern,action_minute,event_seconds,loss_x,loss_y,transition_x,transition_y,turnover_event_source,turnover_event_id,turnover_action_type,turnover_player,turnover_event_team,turnover_event_order,turnover_event_seconds,turnover_x,turnover_y,raw_event_x,raw_event_y,pass_outcome,dribble_outcome,ball_receipt_outcome,duel_type,duel_outcome,interception_outcome,50_50_outcome,clearance_aerial_won,clearance_body_part,ball_recovery_offensive,block_deflection,block_offensive,block_save_block,under_pressure,counterpress,shot_within_5s,seconds_to_shot,shot_player,shot_outcome,shot_statsbomb_xg,losing_team_score_state,losing_team_score_diff,losing_team_players,transition_team_players,transition_team_player_diff,is_numerical_equality
0,3895250,1,Bayer Leverkusen,FSV Mainz 05,8,9,From Keeper,Regular Play,4.08,245.0,7.0,66.0,113.0,14.0,attacking_event,bbd507c2-3e70-48ec-bbbb-006428267e1c,Ball Receipt*,Jonathan Tah,Bayer Leverkusen,115.0,245.0,7.0,66.0,7.0,66.0,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,winning,1,11,11,0,True
1,3895250,1,Bayer Leverkusen,FSV Mainz 05,10,11,From Free Kick,Regular Play,5.22,313.0,38.8,46.6,81.2,33.4,defensive_event,f9db3df4-9bae-43f3-a735-bcb10c61f0f8,Block,Leandro Barreiro Martins,FSV Mainz 05,159.0,303.0,81.2,33.4,81.2,33.4,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,winning,1,11,11,0,True
2,3895250,1,FSV Mainz 05,Bayer Leverkusen,13,14,From Free Kick,From Counter,6.30,378.0,113.4,38.6,6.6,41.4,defensive_event,4805e2a9-47ac-44b0-a338-0315fd96644d,Clearance,Alejandro Grimaldo García,Bayer Leverkusen,216.0,375.0,6.6,41.4,6.6,41.4,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,Right Foot,<NA>,<NA>,NaN,NaN,True,NaN,0,NaN,NaN,NaN,NaN,losing,-1,11,11,0,True
3,3895250,1,Bayer Leverkusen,FSV Mainz 05,14,15,From Counter,Regular Play,6.38,383.0,44.1,41.2,75.9,38.8,attacking_event,e05c5dd6-c915-48ae-8fd1-17ad2845fda5,Ball Receipt*,Florian Wirtz,Bayer Leverkusen,222.0,383.0,44.1,41.2,44.1,41.2,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,winning,1,11,11,0,True
4,3895250,1,Bayer Leverkusen,FSV Mainz 05,19,20,From Throw In,Regular Play,9.20,552.0,73.2,60.9,46.8,19.1,attacking_event,a65e2051-fbe7-4a8c-8c0b-4e48ccfdf954,Ball Receipt*,Florian Wirtz,Bayer Leverkusen,286.0,552.0,73.2,60.9,73.2,60.9,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,drawing,0,11,11,0,True


## 7. Dataframes finales

`model_dataset` contiene todas las perdidas en juego y la variable objetivo `shot_within_5s`.

Dejamos solo columnas razonables para empezar una regresion logistica. Evitamos meter demasiadas columnas de depuracion o texto crudo.

`dangerous_losses_5s` contiene solo los positivos: perdidas que terminan en tiro rival en 5 segundos o menos.

In [136]:
model_cols = [
    # Target
    "shot_within_5s",

    # Identificacion minima para revisar filas concretas.
    "match_id", "period", "lost_possession", "transition_possession",
    "losing_team", "transition_team",

    # Variables temporales y de marcador.
    "action_minute",
    "losing_team_score_state", "losing_team_score_diff",

    # Contexto numerico. Esta es mas informativa que solo igualdad numerica:
    # +1 significa que el equipo que roba tiene un jugador mas; -2, dos menos.
    "transition_team_player_diff",
    "losing_team_players", "transition_team_players",

    # Posicion de perdida desde el punto de vista del equipo que pierde.
    "loss_x", "loss_y", "transition_x", "transition_y",

    # Tipo de perdida/accion que origina la transicion.
    "turnover_action_type", "turnover_event_source",

    # Contexto directo del evento.
    "under_pressure", "counterpress",

    # Variables de resultado auxiliares para revisar positivos; no son predictors.
    "seconds_to_shot", "shot_outcome", "shot_statsbomb_xg",
]

model_dataset = model_dataset[[c for c in model_cols if c in model_dataset.columns]].copy()
dangerous_losses_5s = (
    model_dataset[model_dataset["shot_within_5s"] == 1]
    .copy()
    .reset_index(drop=True)
)

print(f"Filas para regresion logistica: {len(model_dataset):,}")
print(f"Positivos con tiro rival en <= {SHOT_WINDOW_SECONDS}s: {len(dangerous_losses_5s):,}")
dangerous_losses_5s


Filas para regresion logistica: 59
Positivos con tiro rival en <= 5s: 2


,shot_within_5s,match_id,period,lost_possession,transition_possession,losing_team,transition_team,action_minute,losing_team_score_state,losing_team_score_diff,transition_team_player_diff,losing_team_players,transition_team_players,loss_x,loss_y,transition_x,transition_y,turnover_action_type,turnover_event_source,under_pressure,counterpress,seconds_to_shot,shot_outcome,shot_statsbomb_xg
0,1,3895250,1,73,74,FSV Mainz 05,Bayer Leverkusen,39.40,drawing,0,0,11,11,8.4,28.3,111.6,51.7,Ball Recovery,next_possession_recovery,NaN,NaN,0.0,Saved,0.344265
1,1,3895250,2,109,110,Bayer Leverkusen,FSV Mainz 05,54.57,drawing,0,0,11,11,17.7,43.1,102.3,36.9,Ball Recovery,next_possession_recovery,NaN,NaN,0.0,Off T,0.400750


In [137]:
model_dataset

,shot_within_5s,match_id,period,lost_possession,transition_possession,losing_team,transition_team,action_minute,losing_team_score_state,losing_team_score_diff,transition_team_player_diff,losing_team_players,transition_team_players,loss_x,loss_y,transition_x,transition_y,turnover_action_type,turnover_event_source,under_pressure,counterpress,seconds_to_shot,shot_outcome,shot_statsbomb_xg
0,0,3895250,1,8,9,Bayer Leverkusen,FSV Mainz 05,4.08,winning,1,0,11,11,7.0,66.0,113.0,14.0,Ball Receipt*,attacking_event,NaN,NaN,NaN,NaN,NaN
1,0,3895250,1,10,11,Bayer Leverkusen,FSV Mainz 05,5.22,winning,1,0,11,11,38.8,46.6,81.2,33.4,Block,defensive_event,NaN,NaN,NaN,NaN,NaN
2,0,3895250,1,13,14,FSV Mainz 05,Bayer Leverkusen,6.30,losing,-1,0,11,11,113.4,38.6,6.6,41.4,Clearance,defensive_event,True,NaN,NaN,NaN,NaN
3,0,3895250,1,14,15,Bayer Leverkusen,FSV Mainz 05,6.38,winning,1,0,11,11,44.1,41.2,75.9,38.8,Ball Receipt*,attacking_event,NaN,NaN,NaN,NaN,NaN
4,0,3895250,1,19,20,Bayer Leverkusen,FSV Mainz 05,9.20,drawing,0,0,11,11,73.2,60.9,46.8,19.1,Ball Receipt*,attacking_event,NaN,NaN,NaN,NaN,NaN
5,0,3895250,1,24,25,Bayer Leverkusen,FSV Mainz 05,12.13,drawing,0,0,11,11,92.1,77.6,27.9,2.4,Ball Receipt*,attacking_event,NaN,NaN,NaN,NaN,NaN
6,0,3895250,1,25,26,FSV Mainz 05,Bayer Leverkusen,12.22,drawing,0,0,11,11,65.4,20.2,54.6,59.8,Ball Receipt*,attacking_event,NaN,NaN,NaN,NaN,NaN
7,0,3895250,1,26,27,Bayer Leverkusen,FSV Mainz 05,13.82,drawing,0,0,11,11,99.1,31.5,20.9,48.5,Interception,defensive_event,NaN,NaN,NaN,NaN,NaN
8,0,3895250,1,33,34,Bayer Leverkusen,FSV Mainz 05,17.78,drawing,0,0,11,11,104.1,40.5,15.9,39.5,Ball Recovery,next_possession_recovery,NaN,NaN,NaN,NaN,NaN
9,0,3895250,1,34,35,FSV Mainz 05,Bayer Leverkusen,17.97,drawing,0,0,11,11,76.4,11.9,43.6,68.1,Ball Receipt*,attacking_event,NaN,NaN,NaN,NaN,NaN


## 8. Comprobaciones rapidas

In [138]:
print("Tipos de perdida detectados")
display(
    model_dataset
    .groupby(["turnover_action_type", "turnover_event_source"], dropna=False)
    .agg(perdidas=("match_id", "count"), tiros_5s=("shot_within_5s", "sum"))
    .assign(pct_tiro_5s=lambda d: (100 * d["tiros_5s"] / d["perdidas"]).round(1))
    .reset_index()
    .sort_values("perdidas", ascending=False)
)

print("Resumen por estado de marcador e igualdad numerica")
display(
    model_dataset
    .groupby(["losing_team_score_state", "transition_team_player_diff"], dropna=False)
    .agg(perdidas=("match_id", "count"), tiros_5s=("shot_within_5s", "sum"))
    .assign(pct_tiro_5s=lambda d: (100 * d["tiros_5s"] / d["perdidas"]).round(1))
    .reset_index()
    .sort_values(["losing_team_score_state", "transition_team_player_diff"])
)


Tipos de perdida detectados


,turnover_action_type,turnover_event_source,perdidas,tiros_5s,pct_tiro_5s
0,Ball Receipt*,attacking_event,19,0,0.0
1,Ball Recovery,next_possession_recovery,14,2,14.3
3,Clearance,defensive_event,7,0,0.0
2,Block,defensive_event,6,0,0.0
4,Dispossessed,attacking_event,4,0,0.0
7,Interception,defensive_event,3,0,0.0
5,Dribble,attacking_event,2,0,0.0
8,Miscontrol,attacking_event,2,0,0.0
6,Duel,defensive_event,1,0,0.0
9,Pass,attacking_event,1,0,0.0


Resumen por estado de marcador e igualdad numerica


,losing_team_score_state,transition_team_player_diff,perdidas,tiros_5s,pct_tiro_5s
0,drawing,0,36,2,5.6
1,losing,0,3,0,0.0
2,losing,1,5,0,0.0
3,winning,-1,8,0,0.0
4,winning,0,7,0,0.0
